In [29]:
# Wstępna analiza danych - Fraud Detection

# 1. Import niezbędnych bibliotek
import pandas as pd
import numpy as np

In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
print("Wczytywanie danych o transakcjach...")
transactions_df = pd.read_json('../data/raw/transactions.json', lines=True)
print(f"Wymiary zbioru transakcji: {transactions_df.shape}")

print("\nWczytywanie danych o sprzedawcach...")
merchants_df = pd.read_csv('../data/raw/merchants.csv')
print(f"Wymiary zbioru sprzedawców: {merchants_df.shape}")

print("\nWczytywanie danych o użytkownikach...")
users_df = pd.read_csv('../data/raw/users.csv')
print(f"Wymiary zbioru użytkowników: {users_df.shape}")


Wczytywanie danych o transakcjach...
Wymiary zbioru transakcji: (500000, 14)

Wczytywanie danych o sprzedawcach...
Wymiary zbioru sprzedawców: (1000, 8)

Wczytywanie danych o użytkownikach...
Wymiary zbioru użytkowników: (20000, 10)


In [32]:
# Lista kolumn do wykluczenia
exclude_columns = {'transaction_id', 'user_id', 'merchant_id'}

# Tworzymy słownik mapujący stare nazwy na nowe (z suffixem "_transaction")
rename_dict = {
    col: f"{col}_transaction" 
    for col in transactions_df.columns 
    if col not in exclude_columns
}

# Zmieniamy nazwy kolumn w DataFrame
transactions_df.rename(columns=rename_dict, inplace=True)

In [33]:
rename_dict = {
    col: f"{col}_merchant"
    for col in merchants_df.columns
    if col != 'merchant_id'
}
merchants_df.rename(columns=rename_dict, inplace=True)

In [34]:
# Lub Metoda 2: Słownik rename
users_df.rename(columns={
    col: f"{col}_user" 
    for col in users_df.columns 
    if col != 'user_id'
}, inplace=True)

In [35]:
# Joining transactions_df with merchants_df on 'merchant_id'
merged_df = transactions_df.merge(merchants_df, on='merchant_id', how='left')

In [36]:
# Joining the resulting dataframe with users_df on 'user_id'
df = merged_df.merge(users_df, on='user_id', how='left')

# Mapowanie dnia roku i minuty dnia na sinus i cosinus
# Prawdopodobnie nie wniesie to żadnej wartości...
df['day_of_year_tt'] = df['timestamp_transaction'].dt.dayofyear
df['minute_of_day_tt'] = df['timestamp_transaction'].dt.hour * 60 + df['timestamp_transaction'].dt.minute

df['sin_day_tt'] = np.sin(2 * np.pi * df['day_of_year_tt'] / 365)
df['cos_day_tt'] = np.cos(2 * np.pi * df['day_of_year_tt'] / 365)

df['sin_minute_tt'] = np.sin(2 * np.pi * df['minute_of_day_tt'] / 1440)
df['cos_minute_tt'] = np.cos(2 * np.pi * df['minute_of_day_tt'] / 1440)

In [37]:
# Kolumny, które da się użyć do modelowania, bez encodingu. Ja bym zrobił One-hot
# Do poczytania: contigency table, chi2 test, cramer's V.
# Brak czasu na szukanie takich driverów...
COLS = [
    "amount_transaction", 
    "is_international_transaction", 
    "session_length_seconds_transaction",
    "is_first_time_merchant_transaction",
    "trust_score_merchant",
    "avg_transaction_amount_merchant",
    "account_age_months_merchant",
    "has_fraud_history_merchant",
    "age_user",
    "sum_of_monthly_installments_user",
    "sum_of_monthly_expenses_user",
    "risk_score_user",
    "sin_day_tt",
    "cos_day_tt",
    "sin_minute_tt",
    "cos_minute_tt",
]

In [38]:
new_df = df.dropna(subset=COLS)

In [39]:
# Podział na zbiór treningowy i testowy
# 80% zbioru treningowego, 20% testowego

X = new_df.sort_values(by='timestamp_transaction')[COLS]
y = new_df[["is_fraud_transaction"]]

X_train_global = X.iloc[:int(len(X) * 0.8)]
X_test_global = X.drop(X_train_global.index)

y_train_global = y.loc[X_train_global.index]
y_test_global = y.drop(y_train_global.index)

In [40]:
# Downsampling zmiennej dominującej.
# Robię to tak, biorę ilość zmiennej FRAUD = 1
# i dzielę na 3 zbiory, losowo.
# Następnie do każdego zbioru tak wybranego losuję zmienną FRAUD = 0.
# Czyli mam 3 foldy, które są zbalansowane.
# Aby wykorzystać więcej data, powtarzam ten proces 3 razy.
# Przez co powstają 3 cross-validation sety
# 1. X - A, Y - B, Z - C
# 2. X - AA, Y - BB, Z - CC
# 3. X - AAA, Y - BBB, Z - CCC
# Jak widać X, Y, Z są stałe, a A, B, C są zmienne.
# Ma to sens???
# Eksperyment.... Se wymyśliłem
# Random search na danych???

FOLDS = 3
MULT_OF_CVS = 3

y_is_fraud = y_train_global.query("is_fraud_transaction == 1")["is_fraud_transaction"]
y_not_fraud = y_train_global.query("is_fraud_transaction == 0")["is_fraud_transaction"]

y_true_length = len(y_is_fraud)
y_false_length = len(y_not_fraud)

y_is_fraud = y_is_fraud.sample(frac=1, random_state=42)
y_not_fraud = y_not_fraud.sample(frac=1, random_state=42)
ranges_true = np.linspace(0, y_true_length, FOLDS+1).astype(int)
ranges_false = np.linspace(0, y_false_length, MULT_OF_CVS+1).astype(int)

y_is_fraud_cv = [y_is_fraud.iloc[ranges_true[i]:ranges_true[i+1]] for i in range(len(ranges_true)-1)]
y_not_fraud_cv = [y_not_fraud.iloc[ranges_false[i]:ranges_false[i+1]] for i in range(len(ranges_false)-1)]

for i in range(MULT_OF_CVS):
    y_not_fraud_cv[i] = [
        y_not_fraud_cv[i].sample(n=len(y_is_fraud_cv[i]), random_state=42)
        for _ in range(MULT_OF_CVS)
    ]

sets = [0] * MULT_OF_CVS
for i in range(MULT_OF_CVS):
    sets[i] = [ 
        pd.concat([y_is_fraud_cv[j], y_not_fraud_cv[j][i]])
        for j in range(FOLDS)
    ]


cvs = {
    f"dataset_{index}": {
        f"cv_{i+1}": [set_[i], X_train_global.loc[set_[i].index]]
        for i in range(FOLDS)
    }
    for index, set_ in enumerate(sets)
}  

In [63]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Obsługa moich wielu foldów
def custom_cv(cvs: dict[str, dict[str, list[pd.DataFrame]]], model):
    accuracies = [[] for _ in range(len(cvs))]
    for index, dataset in enumerate(cvs.values()):
        print(f"\nDataset {index+1}:\n")
        for i in range(len(dataset)):
            X_train = pd.concat([dataset[f"cv_{j+1}"][1] for j in range(len(dataset)) if j != i])
            y_train = pd.concat([dataset[f"cv_{j+1}"][0] for j in range(len(dataset)) if j != i])

            X_test = dataset[f"cv_{i+1}"][1]
            y_test = dataset[f"cv_{i+1}"][0]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            accuracies[index].append(accuracy)
            print(f"CV: {i+1}, Accuracy: {accuracy}")

    overall_accuracy = np.mean(accuracies, axis=1)
    print(f"\nOverall accuracy: {overall_accuracy}")
    best_index = np.argmax(overall_accuracy)
    
    data = list(cvs.values())[best_index]
    X_train = pd.concat([data[f"cv_{j+1}"][1] for j in range(len(data))])
    y_train = pd.concat([data[f"cv_{j+1}"][0] for j in range(len(data))])
    model.fit(X_train, y_train)

    return np.max(overall_accuracy), model

In [64]:
# Grid Search na moim własnym cross-validation
rf = RandomForestClassifier()
parameters = {
    'n_estimators': [100, 200],
    'max_depth': [None],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    #'max_depth': [None, 10, 20],
    #'min_samples_split': [2, 5],
    #'min_samples_leaf': [1, 2],
}

min_accuracy = 0
best_model = None
i = 1
for n_estimators in parameters['n_estimators']:
    for max_depth in parameters['max_depth']:
        for min_samples_split in parameters['min_samples_split']:
            for min_samples_leaf in parameters['min_samples_leaf']:
                    print(f"MODEL: {i}")
                    rf = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                    )
                    accuracy, model = custom_cv(cvs=cvs, model=rf)
                    if accuracy > min_accuracy:
                        min_accuracy = accuracy
                        best_model = model
                        print(f"New best model found with accuracy: {accuracy:.4f}")
                    i += 1

MODEL: 1

Dataset 1:

CV: 1, Accuracy: 0.5368645195634815
CV: 2, Accuracy: 0.5364652648389673
CV: 3, Accuracy: 0.5350869410929737

Dataset 2:

CV: 1, Accuracy: 0.5364652648389673
CV: 2, Accuracy: 0.5379291988288528
CV: 3, Accuracy: 0.5334900638750887

Dataset 3:

CV: 1, Accuracy: 0.5381066453730814
CV: 2, Accuracy: 0.5327388874101677
CV: 3, Accuracy: 0.5355305180979418

Overall accuracy: [0.53613891 0.53596151 0.53545868]
New best model found with accuracy: 0.5361
MODEL: 2

Dataset 1:

CV: 1, Accuracy: 0.5443172744210807
CV: 2, Accuracy: 0.5419661077100524
CV: 3, Accuracy: 0.537304826117814

Dataset 2:

CV: 1, Accuracy: 0.5420991926182238
CV: 2, Accuracy: 0.5401029189956525
CV: 3, Accuracy: 0.5377040454222853

Dataset 3:

CV: 1, Accuracy: 0.541211959897081
CV: 2, Accuracy: 0.5422322775263951
CV: 3, Accuracy: 0.5369056068133428

Overall accuracy: [0.54119607 0.53996872 0.54011661]
New best model found with accuracy: 0.5412


In [67]:
y_pred = best_model.predict(X_test_global)
accuracy = accuracy_score(y_test_global, y_pred)
print(f"\nFinal accuracy on test set: {accuracy:.4f}")

from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(y_test_global, y_pred, target_names=["Not Fraud", "Fraud"]))


Final accuracy on test set: 0.5149

Classification Report:
              precision    recall  f1-score   support

   Not Fraud       0.92      0.52      0.66     91403
       Fraud       0.09      0.49      0.15      8597

    accuracy                           0.51    100000
   macro avg       0.50      0.50      0.40    100000
weighted avg       0.84      0.51      0.62    100000



In [66]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test_global, y_pred).ravel()
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

True Negatives: 47279
False Positives: 44124
False Negatives: 4389
True Positives: 4208
